In [1]:
import brightway2 as bw
from brightway2 import *
import os
import wurst
import time
import openpyxl
import copy
import numpy as np
import pandas as pd
import glob as glob

In [2]:
startTime = time.time() # just to see how much time the code takes to run (this is the start)

In [3]:
bw.projects.set_current('Prospective LCA v5') # set current project

In [4]:
newLocations = {'BR' : 'Brazil',
                'CA' : 'Canada',
                'PL' : 'Central Europe',
                'CN' : 'China',
                'ET' : 'Eastern Africa',
                'IN' : 'India',
                'ID' : 'Indonesia',
                'JP' : 'Japan',
                'KR' : 'Korea',
                'IR' : 'Middle East',
                'MX' : 'Mexico',
                'EG' : 'Northern Africa',
                'AU' : 'Oceania',
                'GT' : 'Rest of Central America',
                'BW' : 'Rest of Southern Africa',
                'CL' : 'Rest of Southern America',
                'PK' : 'Rest of Southern Asia',
                'RU' : 'Russia',
                'ZA' : 'South Africa',
                'PH' : 'South Eastern Asia',
                'UZ' : 'Central Asia',
                'TR' : 'Turkey',
                'UA' : 'Ukraine',
                'US' : 'United States of America',
                'NG' : 'Western Africa',
                'RER' : 'Western Europe'}

In [5]:
allDBNames = list(bw.databases)

In [6]:
ecoinventSSP2DBNames = ['SSP2-Base 2020',
                        'SSP2-Base 2050',
                        'SSP2-RCP19 2050']

In [7]:
imageLocations = {'Global' : 'World',
                'Brazil' : 'BRA',
                'Canada' : 'CAN',
                'Central Europe' : 'CEU',
                'China' : 'CHN',
                'Eastern Africa' : 'EAF',
                'India' : 'INDIA',
                'Indonesia' : 'INDO',
                'Japan' : 'JAP',
                'Korea' : 'KOR',
                'Middle East' : 'ME',
                'Mexico' : 'MEX',
                'Northern Africa' : 'NAF',
                'Oceania' : 'OCE',
                'Rest of Central America' : 'RCAM',
                'Rest of Southern Africa' : 'RSAF',
                'Rest of Southern America' : 'RSAM',
                'Rest of Southern Asia' : 'RSAS',
                'Russia' : 'RUS',
                'South Africa' : 'SAF',
                'South Eastern Asia' : 'SEAS',
                'Central Asia' : 'STAN',
                'Turkey' : 'TUR',
                'Ukraine' : 'UKR',
                'United States of America' : 'USA',
                'Western Africa' : 'WAF',
                'Western Europe' : 'WEU'}

solarLocations = {'Brazil' : 'RoW',
                  'Canada' : 'CA-AB',
                  'Central Europe' : 'RoW',
                  'China' : 'CN-BJ',
                  'Eastern Africa' : 'RoW',
                  'India' : 'RoW',
                  'Indonesia' : 'RoW',
                  'Japan' : 'JP',
                  'Korea' : 'KR',
                  'Middle East' : 'RoW',
                  'Mexico' : 'MX',
                  'Northern Africa' : 'RoW',
                  'Oceania' : 'AU',
                  'Rest of Central America' : 'RoW',
                  'Rest of Southern Africa' : 'RoW',
                  'Rest of Southern America' : 'AR',
                  'Rest of Southern Asia' : 'RoW',
                  'Russia' : 'RoW',
                  'South Africa' : 'RoW',
                  'South Eastern Asia' : 'RoW',
                  'Central Asia' : 'RoW',
                  'Turkey' : 'RoW',
                  'Ukraine' : 'RoW',
                  'United States of America' : 'US-HICC',
                  'Western Africa' : 'RoW',
                  'Western Europe' : 'DE'}

windLocations =  {'Brazil' : 'RoW',
                  'Canada' : 'CA-AB',
                  'Central Europe' : 'BG',
                  'China' : 'CN-AH',
                  'Eastern Africa' : 'RoW',
                  'India' : 'IN-TN',
                  'Indonesia' : 'RoW',
                  'Japan' : 'JP',
                  'Korea' : 'KR',
                  'Middle East' : 'IR',
                  'Mexico' : 'MX',
                  'Northern Africa' : 'RoW',
                  'Oceania' : 'AU',
                  'Rest of Central America' : 'RoW',
                  'Rest of Southern Africa' : 'RoW',
                  'Rest of Southern America' : 'CL',
                  'Rest of Southern Asia' : 'RoW',
                  'Russia' : 'RU',
                  'South Africa' : 'ZA',
                  'South Eastern Asia' : 'RoW',
                  'Central Asia' : 'RoW',
                  'Turkey' : 'TR',
                  'Ukraine' : 'UA',
                  'United States of America' : 'US-ASCC',
                  'Western Africa' : 'RoW',
                  'Western Europe' : 'DE'}

def find_key_by_value(dictionary, value):
    for key, val in dictionary.items():
        if val == value:
            return key
    return None

def find_value_by_key(dictionary, key):
    for k, val in dictionary.items():
        if k == key:
            return val
    return None

In [8]:
"""for ecoinventSSP2DBName in ecoinventSSP2DBNames:
    
    ecoinventSSP2DB = bw.Database('ecoinvent 3.8 cutoff image ' + ecoinventSSP2DBName)
    # mySSP2DBName = ecoinventSSP2DBName.replace('ecoinvent 3.8 cutoff', 'lci-Abhi')
    mySSP2DB = bw.Database('lci-Abhi characterisation factors')
    # print('Started ' + mySSP2DBName)

    locations = imageLocations

    for value, location in locations.items():

        ecoinventLocation = find_key_by_value(newLocations, value)
        windLocation = find_value_by_key(windLocations, value)
        solarLocation = find_value_by_key(solarLocations, value)

        if ecoinventLocation:
            newLoc = ecoinventLocation
            
            solarElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, photovoltaic, 570kWp open ground installation, multi-Si' in activity['name']
                        and 'electricity, low voltage' in activity['reference product']
                        and solarLocation == activity['location']]
            
            onshoreWindElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, wind, >3MW turbine, onshore' in activity['name']
                        and 'electricity, high voltage' in activity['reference product']
                        and windLocation == activity['location']]
                
        else:
            newLoc = 'GLO'

            solarElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, photovoltaic, 570kWp open ground installation, multi-Si' in activity['name']
                        and 'electricity, low voltage' in activity['reference product']
                        and 'RoW' in activity['location']]
            
            onshoreWindElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, wind, >3MW turbine, onshore' in activity['name']
                        and 'electricity, high voltage' in activity['reference product']
                        and 'RoW' in activity['location']]

        solarElectActCopy = solarElecAct[0].copy(database = mySSP2DB.name, location = newLoc, name = 'solar ' + ecoinventSSP2DBName)
        onshoreWindElecActCopy = onshoreWindElecAct[0].copy(database = mySSP2DB.name, location = newLoc, name = 'onshore wind ' + ecoinventSSP2DBName)
        
    # print('Finished ', mySSP2DBName)"""

"for ecoinventSSP2DBName in ecoinventSSP2DBNames:\n    \n    ecoinventSSP2DB = bw.Database('ecoinvent 3.8 cutoff image ' + ecoinventSSP2DBName)\n    # mySSP2DBName = ecoinventSSP2DBName.replace('ecoinvent 3.8 cutoff', 'lci-Abhi')\n    mySSP2DB = bw.Database('lci-Abhi characterisation factors')\n    # print('Started ' + mySSP2DBName)\n\n    locations = imageLocations\n\n    for value, location in locations.items():\n\n        ecoinventLocation = find_key_by_value(newLocations, value)\n        windLocation = find_value_by_key(windLocations, value)\n        solarLocation = find_value_by_key(solarLocations, value)\n\n        if ecoinventLocation:\n            newLoc = ecoinventLocation\n            \n            solarElecAct = [activity for activity in ecoinventSSP2DB\n                        if 'electricity production, photovoltaic, 570kWp open ground installation, multi-Si' in activity['name']\n                        and 'electricity, low voltage' in activity['reference product']\n     

In [9]:
databaseNames = bw.databases
myDatabaseNames = []
for databaseName in databaseNames:
    if 'Abhi' in databaseName and 'image' not in databaseName:
        myDatabaseNames.append(databaseName)

In [10]:
myDatabaseNames

[]

In [11]:
methodsDict = {'climate change' : ('IPCC 2013', 'climate change', 'GWP 20a, incl. H and bio CO2')}    

In [12]:
methodsList = []
solarResultsFileNames = []
onshoreWindResultsFileNames = []
for keys, values in methodsDict.items():  
    methodsList.append(values)
    solarResultsFileNames.append('solar ' + keys + ' results.xlsx')
    onshoreWindResultsFileNames.append('onshore wind ' + keys + ' results.xlsx')

allResultsFileNames = solarResultsFileNames + onshoreWindResultsFileNames

In [13]:
def lca_calculations(activities, methodsList):
    results = np.zeros((len(activities), len(methodsList)))
    lca = LCA({activities[0] : 1}, method = methodsList[0]) # LCA object which will do all calculating
    lca.lci() # calculate inventory once to load all database data
    lca.decompose_technosphere() # keep the LU factorized matrices for faster calculations
    lca.lcia() # load method data
    characterizationMatrices = []
    
    for method in methodsList:
        lca.switch_method(method)
        characterizationMatrices.append(lca.characterization_matrix.copy())
        
    for first, activity in enumerate(activities):
        lca.redo_lci({activity : 1})
        
        for second, matrix in enumerate(characterizationMatrices):
            results[first, second] = (matrix * lca.inventory).sum()
            
    return results

In [14]:
def check_or_create_excel_file(filePath):
    if not os.path.exists(filePath):
        wb = openpyxl.Workbook()
        wb.save(filePath)
        print(f'Excel file created at {filePath}')

In [15]:
def delete_first_sheet_from_excel_files(filePath):
    wb = openpyxl.load_workbook(filePath)
    if wb.sheetnames[0] == 'Sheet':  # check if the workbook has any sheets
        firstSheet = wb.sheetnames[0]  # get the name of the first sheet
        wb.remove(wb[firstSheet])  # remove the first sheet
        wb.save(filePath)

In [16]:
"""for myDatabaseName in myDatabaseNames:
    allInventories = []
    allNames = []
    allLocations = []
    myDatabase = bw.Database(myDatabaseName)
    for activity in myDatabase:
        if 'solar' in activity['name']:
            allInventories.append(activity)
            allNames.append(activity['name'])
            allLocations.append(activity['location'])
    lcaScores = lca_calculations(allInventories, methodsList)
    for i in range(len(solarResultsFileNames)):
        solarResutlsFilePath = os.path.join('..', 'Results', 'Characterisation factors', solarResultsFileNames[i])
        check_or_create_excel_file(solarResutlsFilePath)
        with pd.ExcelWriter(solarResutlsFilePath, engine = 'openpyxl', mode = 'a') as writer:
            lcaScoresSpecific = lcaScores[:, i].ravel().tolist()
            lcaScoresDF = pd.DataFrame({'Activity' : allNames, 'Location' : allLocations, 'Value' : lcaScoresSpecific})
            lcaScoresDFPivot = lcaScoresDF.pivot(index = 'Activity', columns = 'Location', values = 'Value')
            if myDatabaseName in writer.book.sheetnames:
                writer.book.remove(writer.book[myDatabaseName])
            lcaScoresDFPivot.to_excel(writer, sheet_name = myDatabaseName.replace("lci-Abhi ", ""))"""

'for myDatabaseName in myDatabaseNames:\n    allInventories = []\n    allNames = []\n    allLocations = []\n    myDatabase = bw.Database(myDatabaseName)\n    for activity in myDatabase:\n        if \'solar\' in activity[\'name\']:\n            allInventories.append(activity)\n            allNames.append(activity[\'name\'])\n            allLocations.append(activity[\'location\'])\n    lcaScores = lca_calculations(allInventories, methodsList)\n    for i in range(len(solarResultsFileNames)):\n        solarResutlsFilePath = os.path.join(\'..\', \'Results\', \'Characterisation factors\', solarResultsFileNames[i])\n        check_or_create_excel_file(solarResutlsFilePath)\n        with pd.ExcelWriter(solarResutlsFilePath, engine = \'openpyxl\', mode = \'a\') as writer:\n            lcaScoresSpecific = lcaScores[:, i].ravel().tolist()\n            lcaScoresDF = pd.DataFrame({\'Activity\' : allNames, \'Location\' : allLocations, \'Value\' : lcaScoresSpecific})\n            lcaScoresDFPivot = lc

In [17]:
"""for myDatabaseName in myDatabaseNames:
    allInventories = []
    allNames = []
    allLocations = []
    myDatabase = bw.Database(myDatabaseName)
    for activity in myDatabase:
        if 'onshore wind' in activity['name']:
            allInventories.append(activity)
            allNames.append(activity['name'])
            allLocations.append(activity['location'])
    lcaScores = lca_calculations(allInventories, methodsList)
    for i in range(len(onshoreWindResultsFileNames)):
        onshoreWindResultsFilePath = os.path.join('..', 'Results', 'Characterisation factors', onshoreWindResultsFileNames[i])
        check_or_create_excel_file(onshoreWindResultsFilePath)
        with pd.ExcelWriter(onshoreWindResultsFilePath, engine = 'openpyxl', mode = 'a') as writer:
            lcaScoresSpecific = lcaScores[:, i].ravel().tolist()
            lcaScoresDF = pd.DataFrame({'Activity' : allNames, 'Location' : allLocations, 'Value' : lcaScoresSpecific})
            lcaScoresDFPivot = lcaScoresDF.pivot(index = 'Activity', columns = 'Location', values = 'Value')
            if myDatabaseName in writer.book.sheetnames:
                writer.book.remove(writer.book[myDatabaseName])
            lcaScoresDFPivot.to_excel(writer, sheet_name = myDatabaseName.replace("lci-Abhi ", ""))"""

'for myDatabaseName in myDatabaseNames:\n    allInventories = []\n    allNames = []\n    allLocations = []\n    myDatabase = bw.Database(myDatabaseName)\n    for activity in myDatabase:\n        if \'onshore wind\' in activity[\'name\']:\n            allInventories.append(activity)\n            allNames.append(activity[\'name\'])\n            allLocations.append(activity[\'location\'])\n    lcaScores = lca_calculations(allInventories, methodsList)\n    for i in range(len(onshoreWindResultsFileNames)):\n        onshoreWindResultsFilePath = os.path.join(\'..\', \'Results\', \'Characterisation factors\', onshoreWindResultsFileNames[i])\n        check_or_create_excel_file(onshoreWindResultsFilePath)\n        with pd.ExcelWriter(onshoreWindResultsFilePath, engine = \'openpyxl\', mode = \'a\') as writer:\n            lcaScoresSpecific = lcaScores[:, i].ravel().tolist()\n            lcaScoresDF = pd.DataFrame({\'Activity\' : allNames, \'Location\' : allLocations, \'Value\' : lcaScoresSpecific

In [18]:
"""allResultsFileNames = glob.glob(os.path.join('..', 'Results', 'Characterisation factors', '*.xls*'), recursive = True)
for filePath in allResultsFileNames:
    delete_first_sheet_from_excel_files(filePath)"""

"allResultsFileNames = glob.glob(os.path.join('..', 'Results', 'Characterisation factors', '*.xls*'), recursive = True)\nfor filePath in allResultsFileNames:\n    delete_first_sheet_from_excel_files(filePath)"

In [19]:
endTime = time.time() # end time
elapsedTime = endTime - startTime # calculate elapsed time
print(f'Elapsed time: {elapsedTime/3600:.2f} hours') 

Elapsed time: 0.00 hours
